# Colab 02 (Full): Step LoRA Training (Inline)
- `unified_trainer.py` JSSP step 학습 워크플로우를 노트북형으로 전체화
- HF/local dataset source 지원
- resume_from_checkpoint / shuffle_data / output_dir 자동생성 / 업로드 포함


In [ ]:
!pip -q install -U unsloth
!pip -q install -U "transformers==4.57.6" huggingface_hub datasets trl peft accelerate bitsandbytes wandb


## 1) 설정


In [ ]:
import os
import csv
import random
from functools import partial
from pathlib import Path
import math
import torch
import wandb
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download, HfApi, upload_folder
from unsloth import FastLanguageModel

######################
# HUGGING FACE TOKEN #
######################
HF_TOKEN = ''
######################

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = ''
if not HF_TOKEN:
    print('HF_TOKEN is empty. Fill the block above or set a Colab secret/env var before Hugging Face operations.')
else:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print('HF login ready')

CFG = {
    # model
    'max_seq_length': 40000,
    # 'model_type': 'llama8b',
    'model_type': 'llama8b',
    # 'model_type': 'qwen25_7b',
    'dtype': 'bfloat16',
    'load_in_4bit': True,

    # LoRA
    'lora_r': 64,
    'lora_alpha': 128,
    'lora_dropout': 0.0,
    'bias': 'none',
    'use_gradient_checkpointing': 'unsloth',
    'random_state': 42,
    'use_rslora': True,
    'loftq_config': None,

    # train hparams
    'per_device_train_batch_size': 48,
    'gradient_accumulation_steps': 1,
    'per_device_eval_batch_size': 8,
    'num_train_epochs': 2,
    'learning_rate': 2e-4,
    'save_steps': 1000,
    'save_strategy': 'steps',
    'save_total_limit': 20,
    'logging_steps': 10,
    'eval_steps': 1000,
    'warmup_steps': 1000,
    'optim': 'adamw_8bit',
    'weight_decay': 0.01,
    'lr_scheduler_type': 'linear',
    'seed': 42,
    'group_by_length': True,
    'dataset_num_proc': 16,

    # speed controls
    'enable_eval': False,
    # 'max_train_samples': None,
    'max_train_samples': 1000000,
    'max_eval_samples': 512,
    'eval_subset_seed': 42,

    # task flags (unified_trainer.py 호환)
    'train_jssp': True,
    'train_fssp': False,
    'train_vrp_tsp': False,
    'train_knapsack': False,
    'train_binpack': False,
    'train_lm_head': False,
    'train_embed_tokens': False,

    # environment mode
    'env_mode': 'serial',  # serial | dispatch

    # adapter role
    'adapter_role': 'policy',  # policy | reason | mixed
    'action_code_width': 4,
    'action_code_cap': 9999,
    'step_supervision_mode': 'action_only',  # resolved automatically from adapter_role

    # unified_trainer.py 옵션 동등화
    'step_dataset_path': None,
    'fp16': None,
    'bf16': None,
    'dataloader_num_workers': 0,
    'evaluation_strategy': 'steps',
    'load_best_model_at_end': False,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'remove_unused_columns': False,
    'report_to': 'wandb',
    'run_name': None,

    # dataset source
    'dataset_source': 'hf',
    'step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'policy_step_dataset_hf_repo': 'HYUNJINI/jssp_policy_step_train_all_v1',
    'policy_step_dataset_hf_file': 'train_data/jssp_step_train_policy.jsonl',
    'policy_step_dataset_local_path': '/content/jssp_step_train_policy.jsonl',
    'reason_step_dataset_hf_repo': 'HYUNJINI/jssp_reason_step_train_all_v1',
    'reason_step_dataset_hf_file': 'train_data/jssp_step_train_reason.jsonl',
    'reason_step_dataset_local_path': '/content/jssp_step_train_reason.jsonl',
    'mixed_step_dataset_hf_repo': 'HYUNJINI/jssp_mixed_step_train_all_v1',
    'mixed_step_dataset_hf_file': 'train_data/jssp_step_train_with_reason.jsonl',
    'mixed_step_dataset_local_path': '/content/jssp_step_train_with_reason.jsonl',
    'policy_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_policy_step_train_dispatch_v1',
    'policy_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_policy_dispatch.jsonl',
    'policy_step_dataset_local_path_dispatch': '/content/jssp_step_train_policy_dispatch.jsonl',
    'reason_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_reason_step_train_dispatch_v1',
    'reason_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_reason_dispatch.jsonl',
    'reason_step_dataset_local_path_dispatch': '/content/jssp_step_train_reason_dispatch.jsonl',
    'mixed_step_dataset_hf_repo_dispatch': 'HYUNJINI/jssp_mixed_step_train_dispatch_v1',
    'mixed_step_dataset_hf_file_dispatch': 'train_data/jssp_step_train_with_reason_dispatch.jsonl',
    'mixed_step_dataset_local_path_dispatch': '/content/jssp_step_train_with_reason_dispatch.jsonl',

    # data options
    # 'shuffle_data': False,
    'shuffle_data': True,
    'shuffle_seed': 42,

    # output
    'output_accord': False,
    'output_list_of_lists': False,
    'output_dir': None,

    # resume
    'resume_from_checkpoint': None,

    # wandb
    'enable_wandb': False,
    'wandb_project': None,

    # optional upload
    'upload_after_train': False,
    'upload_on_save': True,
    'hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'policy_hf_model_repo_out': 'HYUNJINI/jssp_policy_llama8b_step_r64_ep2',
    'reason_hf_model_repo_out': 'HYUNJINI/jssp_reason_llama8b_step_r64_ep2',
    'upload_source': 'latest_checkpoint',  # final | latest_checkpoint | checkpoint_tag | output_dir
    'checkpoint_tag': 'checkpoint-200',
}

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'llama1b': 'unsloth/Llama-3.1-8B-Instruct',
    'qwen25_7b': 'unsloth/Qwen3.5-9B-Base',
    'qwen25_7b_math': 'unsloth/Qwen2.5-Math-7B-Instruct-bnb-4bit',
    'qwen25_14b': 'unsloth/Qwen2.5-14B-Instruct-unsloth-bnb-4bit',
    'deepseek_8b': 'unsloth/DeepSeek-R1-Distill-Llama-8B-unsloth-bnb-4bit',
}

print(CFG)


## 2) 유틸 함수 (inline)


In [ ]:

# --- Inline helpers aligned with source plain-text <Axxxx> path ---
import csv
import math
import os
import random
from functools import partial
from pathlib import Path
from typing import Optional

import numpy as np
import torch
from datasets import Dataset, load_dataset
from huggingface_hub import HfApi, upload_folder
from peft import PeftModel
from transformers import TrainerCallback

def create_step_prompt_formats(example, tokenizer, step_supervision_mode="action_only"):
    """
    Creates step-by-step JSSP prompt formats for model training.

    Expected input keys:
        - state_text
        - target_text
    """
    state_text = str(example.get("state_text", ""))
    reason_input_text = str(example.get("reason_input_text", ""))
    target_text = str(example.get("target_text", ""))
    reason_target_text = str(example.get("reason_target_text", "")).strip()

    if step_supervision_mode not in {"action_only", "action_reason", "reason_only"}:
        raise ValueError(
            f"Unsupported step_supervision_mode={step_supervision_mode}. "
            "Use one of: action_only, action_reason, reason_only."
        )

    if step_supervision_mode == "reason_only":
        user_content = reason_input_text or state_text
        if not user_content:
            raise ValueError("Missing 'reason_input_text'/'state_text' for reason dataset sample.")
        if not reason_target_text:
            legacy_reason_text = str(example.get("target_reason_text", "")).strip()
            if legacy_reason_text:
                reason_target_text = legacy_reason_text
        if not reason_target_text:
            raise ValueError("Missing 'reason_target_text' for reason dataset sample.")
        assistant_content = reason_target_text
        system_content = (
            "You are an expert JSSP scheduling analyst. "
            "Explain a fixed already-selected action. "
            "Primary objective context is final makespan (Cmax). "
            "Do not output a new action. "
            "Output format:\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
    elif step_supervision_mode == "action_reason":
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        target_action_reason_text = str(example.get("target_action_reason_text", "")).strip()
        if target_action_reason_text:
            assistant_content = target_action_reason_text
        else:
            # Backward compatible fallback for older datasets.
            assistant_content = (
                f"{target_text}\n{reason_target_text}" if reason_target_text else target_text
            )
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code and explain briefly. "
            "Output format:\n"
            "<Axxxx>\n"
            "Reason: ...\n"
            "Not chosen:\n"
            "- <Axxxx>: ..."
        )
        user_content = state_text
    else:
        if not state_text:
            raise ValueError("Missing 'state_text' for step dataset sample.")
        if not target_text:
            raise ValueError("Missing 'target_text' for step dataset sample.")
        assistant_content = target_text
        system_content = (
            "You are an expert JSSP scheduler. "
            "Primary objective: minimize final makespan (Cmax). "
            "At each step, choose exactly one feasible action code. "
            "Always output exactly one code in this format: <Axxxx>."
        )
        user_content = state_text

    messages = [
        {
            "role": "system",
            "content": system_content,
        },
        {
            "role": "user",
            "content": user_content,
        },
        {
            "role": "assistant",
            "content": assistant_content,
        },
    ]

    example["text"] = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return example

MODEL_MAP = {
    'llama8b': 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit',
    'qwen35_9b': 'unsloth/Qwen3.5-9B',
}


def resolve_task_type(cfg):
    if cfg.get('train_jssp', False):
        return 'jssp'
    if cfg.get('train_fssp', False):
        return 'fssp'
    if cfg.get('train_vrp_tsp', False):
        return 'vrp_tsp'
    if cfg.get('train_knapsack', False):
        return 'knapsack'
    if cfg.get('train_binpack', False):
        return 'binpack'
    return 'generic'


def resolve_output_dir(cfg, task_type):
    if cfg.get('output_dir'):
        return cfg['output_dir']
    model_key = cfg['model_type']
    supervision = cfg.get('step_supervision_mode', 'default')
    env_mode = str(cfg.get('env_mode', 'serial')).lower()
    rank = cfg['lora_r']
    epochs = cfg['num_train_epochs']
    return f"/content/outputs/{task_type}_{model_key}_{env_mode}_{supervision}_r{rank}_ep{epochs}"


def resolve_step_supervision_mode(cfg):
    role = cfg.get('adapter_role', 'policy')
    if role == 'policy':
        return 'action_only'
    if role == 'reason':
        return 'reason_only'
    return 'action_reason'


def resolve_step_dataset_path(cfg):
    role = cfg.get('adapter_role', 'policy')
    source = cfg.get('dataset_source', 'hf')
    if role == 'policy':
        repo = cfg.get('policy_step_dataset_hf_repo')
        file = cfg.get('policy_step_dataset_hf_file')
        local = cfg.get('policy_step_dataset_local_path')
    elif role == 'reason':
        repo = cfg.get('reason_step_dataset_hf_repo')
        file = cfg.get('reason_step_dataset_hf_file')
        local = cfg.get('reason_step_dataset_local_path')
    else:
        repo = cfg.get('mixed_step_dataset_hf_repo')
        file = cfg.get('mixed_step_dataset_hf_file')
        local = cfg.get('mixed_step_dataset_local_path')

    if source == 'local':
        return os.path.expanduser(local)

    ds = load_dataset('json', data_files={'train': f'hf://datasets/{repo}/{file}'})
    cache_files = ds['train'].cache_files
    if not cache_files:
        raise RuntimeError('No cache files found for dataset split.')
    return cache_files[0]['filename']


def load_step_jsonl_dataset(path):
    return load_dataset('json', data_files={'train': path})['train']


def maybe_shuffle_dataset(dataset, cfg):
    if cfg.get('shuffle_data', False):
        seed = int(cfg.get('shuffle_seed', 42))
        print(f'shuffle enabled (seed={seed})')
        return dataset.shuffle(seed=seed)
    print('shuffle disabled')
    return dataset


def cap_dataset_before_map(dataset, cfg, eval_ratio=0.05):
    max_train_samples = cfg.get('max_train_samples')
    if max_train_samples is None:
        return dataset
    required_rows = int(math.ceil(int(max_train_samples) / max(1e-9, (1.0 - float(eval_ratio)))))
    required_rows = min(required_rows, len(dataset))
    print(f'pre-map sample cap: {required_rows}')
    return dataset.select(range(required_rows))


def split_train_eval(dataset, cfg):
    eval_ratio = 0.05
    split = dataset.train_test_split(test_size=eval_ratio, seed=int(cfg.get('eval_subset_seed', 42)))
    train_dataset = split['train']
    eval_dataset = split['test']
    if cfg.get('max_train_samples') is not None:
        train_dataset = train_dataset.select(range(min(int(cfg['max_train_samples']), len(train_dataset))))
    if cfg.get('max_eval_samples') is not None:
        eval_dataset = eval_dataset.select(range(min(int(cfg['max_eval_samples']), len(eval_dataset))))
    return train_dataset, eval_dataset


def build_model_name(cfg):
    return MODEL_MAP[cfg['model_type']]


def create_repo_if_needed(api, repo_id, repo_type, private=False):
    api.create_repo(repo_id=repo_id, repo_type=repo_type, private=private, exist_ok=True)


def resolve_upload_dir(output_dir: Path, source: str, checkpoint_tag: Optional[str]):
    if source == 'latest_checkpoint':
        checkpoints = sorted(
            [p for p in output_dir.glob('checkpoint-*') if p.is_dir()],
            key=lambda p: int(p.name.split('-')[-1]),
        )
        if not checkpoints:
            raise FileNotFoundError(f'No checkpoint directories found under {output_dir}')
        return checkpoints[-1]
    if source == 'checkpoint_tag':
        p = output_dir / str(checkpoint_tag)
        if not p.exists():
            raise FileNotFoundError(f'checkpoint folder not found: {p}')
        return p
    p = output_dir / 'final'
    if not p.exists():
        raise FileNotFoundError(f'final folder not found: {p}')
    return p


class UploadCheckpointToHFCallback(TrainerCallback):
    def __init__(self, repo_id, token, output_dir, upload_source='latest_checkpoint'):
        self.repo_id = repo_id
        self.token = token
        self.output_dir = Path(output_dir)
        self.upload_source = upload_source
        self.api = HfApi(token=token)
        create_repo_if_needed(self.api, repo_id, 'model', private=False)

    def on_save(self, args, state, control, **kwargs):
        checkpoint_dir = self.output_dir / f'checkpoint-{state.global_step}'
        if not checkpoint_dir.exists():
            return control
        print(f'[HF upload] checkpoint saved: {checkpoint_dir}')
        upload_folder(
            repo_id=self.repo_id,
            repo_type='model',
            folder_path=str(checkpoint_dir),
            path_in_repo=checkpoint_dir.name,
            token=self.token,
            commit_message=f'Upload {checkpoint_dir.name}',
        )
        print(f'[HF upload] uploaded: {self.repo_id}/{checkpoint_dir.name}')
        return control


def print_number_of_trainable_model_parameters(model):
    trainable = 0
    total = 0
    for p in model.parameters():
        n = p.numel()
        total += n
        if p.requires_grad:
            trainable += n
    ratio = (100.0 * trainable / total) if total else 0.0
    print(f'총 학습가능 매개변수: {trainable:,}개')
    print(f'총 매개변수: {total:,}개')
    print(f'학습가능 매개변수 비율: {ratio:.2f}%')


## 3) 학습 실행 (full)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

base_model_name = MODEL_MAP[CFG['model_type']]
print('base model:', base_model_name)

adapter_role = str(CFG.get('adapter_role', 'policy')).lower()
resolved_step_supervision_mode = {'policy': 'action_only', 'reason': 'reason_only', 'mixed': 'action_reason'}.get(adapter_role)
if resolved_step_supervision_mode is None:
    raise ValueError(f"Unsupported CFG['adapter_role']={adapter_role}")
CFG['step_supervision_mode'] = resolved_step_supervision_mode
if adapter_role == 'policy':
    CFG['hf_model_repo_out'] = CFG.get('policy_hf_model_repo_out', CFG['hf_model_repo_out'])
elif adapter_role == 'reason':
    CFG['hf_model_repo_out'] = CFG.get('reason_hf_model_repo_out', CFG['hf_model_repo_out'])
print('adapter_role:', adapter_role)
print('resolved step supervision mode:', resolved_step_supervision_mode)

task_type = resolve_task_type(CFG)
output_dir = Path(resolve_output_dir(CFG, task_type))
output_dir.mkdir(parents=True, exist_ok=True)

if CFG.get('step_dataset_path'):
    step_dataset_path = os.path.expanduser(CFG['step_dataset_path'])
else:
    env_mode = str(CFG.get('env_mode', 'serial')).lower()
    env_suffix = '_dispatch' if env_mode == 'dispatch' else ''
    if adapter_role == 'reason':
        dataset_repo = CFG.get(f'reason_step_dataset_hf_repo{env_suffix}', CFG.get('reason_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'reason_step_dataset_hf_file{env_suffix}', CFG.get('reason_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'reason_step_dataset_local_path{env_suffix}', CFG.get('reason_step_dataset_local_path', CFG['step_dataset_local_path']))
    elif adapter_role == 'mixed':
        dataset_repo = CFG.get(f'mixed_step_dataset_hf_repo{env_suffix}', CFG.get('mixed_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'mixed_step_dataset_hf_file{env_suffix}', CFG.get('mixed_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'mixed_step_dataset_local_path{env_suffix}', CFG.get('mixed_step_dataset_local_path', CFG['step_dataset_local_path']))
    else:
        dataset_repo = CFG.get(f'policy_step_dataset_hf_repo{env_suffix}', CFG.get('policy_step_dataset_hf_repo', CFG['step_dataset_hf_repo']))
        dataset_file = CFG.get(f'policy_step_dataset_hf_file{env_suffix}', CFG.get('policy_step_dataset_hf_file', CFG['step_dataset_hf_file']))
        dataset_local_path = CFG.get(f'policy_step_dataset_local_path{env_suffix}', CFG.get('policy_step_dataset_local_path', CFG['step_dataset_local_path']))

    if CFG['dataset_source'] == 'hf':
        step_dataset_path = hf_hub_download(
            repo_id=dataset_repo,
            repo_type='dataset',
            filename=dataset_file,
            token=HF_TOKEN,
        )
    elif CFG['dataset_source'] == 'local':
        step_dataset_path = dataset_local_path
    else:
        raise ValueError("CFG['dataset_source'] must be 'hf' or 'local'")

print('step dataset:', step_dataset_path)

dtype = torch.bfloat16 if CFG['dtype'] == 'bfloat16' else torch.float16
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=base_model_name,
    max_seq_length=CFG['max_seq_length'],
    load_in_4bit=CFG['load_in_4bit'],
    dtype=dtype,
    local_files_only=False,
)

modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
if CFG['train_lm_head']:
    modules.append('lm_head')
if CFG['train_embed_tokens']:
    modules.append('embed_tokens')
model = FastLanguageModel.get_peft_model(
    model,
    r=CFG['lora_r'],
    target_modules=modules,
    lora_alpha=CFG['lora_alpha'],
    lora_dropout=CFG['lora_dropout'],
    bias=CFG['bias'],
    use_rslora=CFG['use_rslora'],
    use_gradient_checkpointing=CFG['use_gradient_checkpointing'],
    random_state=CFG['random_state'],
    loftq_config=CFG['loftq_config'],
)
print_number_of_trainable_model_parameters(model)

from datasets import Dataset
import json

# NOTE:
# action_code_to_job has dynamic keys per row (e.g., <A1234>, <A9876>),
# which breaks fixed-schema casting in load_dataset('json').
# We therefore read JSONL directly and keep only training-required fields.
REQUIRED_STEP_KEYS = [
    'state_text',
    'reason_input_text',
    'target_text',
    'target_action_reason_text',
    'target_reason_text',
    'reason_target_text',
    'feature_schema_version',
    'num_jobs',
    'num_machines',
    'total_steps',
    'step_idx',
]


def _normalize_step_row(row):
    out = {}
    out['state_text'] = str(row.get('state_text', ''))
    out['reason_input_text'] = str(row.get('reason_input_text', ''))
    out['target_text'] = str(row.get('target_text', ''))
    out['target_action_reason_text'] = str(row.get('target_action_reason_text', ''))
    out['target_reason_text'] = str(row.get('target_reason_text', ''))
    out['reason_target_text'] = str(row.get('reason_target_text', ''))
    out['feature_schema_version'] = str(row.get('feature_schema_version', 'unknown'))
    out['num_jobs'] = int(row.get('num_jobs', 0))
    out['num_machines'] = int(row.get('num_machines', 0))
    out['total_steps'] = int(row.get('total_steps', 0))
    out['step_idx'] = int(row.get('step_idx', 0))
    return out


def _iter_step_rows(path):
    path = os.path.expanduser(str(path))
    with open(path, 'r', encoding='utf-8') as f:
        # Detect JSON array vs JSONL
        first_non_ws = None
        while True:
            ch = f.read(1)
            if ch == '':
                break
            if not ch.isspace():
                first_non_ws = ch
                break
        f.seek(0)

        if first_non_ws == '[':
            data = json.load(f)
            if not isinstance(data, list):
                raise ValueError('JSON input must be a list when using array format.')
            for row in data:
                yield _normalize_step_row(row)
        else:
            for line_idx, line in enumerate(f, start=1):
                line = line.strip()
                if not line:
                    continue
                try:
                    row = json.loads(line)
                except Exception as e:
                    raise ValueError(f'Invalid JSONL at line {line_idx}: {e}')
                yield _normalize_step_row(row)


dataset = Dataset.from_generator(_iter_step_rows, gen_kwargs={'path': step_dataset_path})
print(f'raw dataset rows: {len(dataset):,}')

if CFG['shuffle_data']:
    print(f"shuffle enabled (seed={CFG['shuffle_seed']})")
    dataset = dataset.shuffle(seed=CFG['shuffle_seed'])
else:
    print('shuffle disabled')

test_size = 0.05
pre_map_cap_candidates = []
if CFG.get('max_train_samples') is not None:
    pre_map_cap_candidates.append(math.ceil(int(CFG['max_train_samples']) / (1.0 - test_size)))
if CFG.get('max_eval_samples') is not None and bool(CFG.get('enable_eval', True)):
    pre_map_cap_candidates.append(math.ceil(int(CFG['max_eval_samples']) / test_size))
pre_map_cap = max(pre_map_cap_candidates) if pre_map_cap_candidates else None
if pre_map_cap is not None:
    pre_map_cap = min(int(pre_map_cap), len(dataset))
    print('pre-map sample cap:', pre_map_cap)
    dataset = dataset.select(range(pre_map_cap))

fmt = partial(create_step_prompt_formats, tokenizer=tokenizer, step_supervision_mode=resolved_step_supervision_mode)
step_map_num_proc = max(1, int(CFG.get('dataset_num_proc', 1)))
print('step prompt map num_proc:', step_map_num_proc)
dataset = dataset.map(fmt, num_proc=step_map_num_proc)

split_dataset = dataset.train_test_split(test_size=test_size, seed=CFG['seed'])
train_dataset = split_dataset['train']
eval_dataset = split_dataset['test']

if CFG.get('max_train_samples') is not None:
    train_cap = min(int(CFG['max_train_samples']), len(train_dataset))
    train_dataset = train_dataset.select(range(train_cap))

if CFG.get('max_eval_samples') is not None:
    eval_cap = min(int(CFG['max_eval_samples']), len(eval_dataset))
    eval_dataset = eval_dataset.shuffle(seed=int(CFG.get('eval_subset_seed', 42))).select(range(eval_cap))

enable_eval = bool(CFG.get('enable_eval', True))
eval_strategy_value = CFG['evaluation_strategy'] if enable_eval else 'no'
eval_steps_value = CFG['eval_steps'] if enable_eval else None
eval_dataset_for_trainer = eval_dataset if enable_eval else None

print('train rows:', len(train_dataset))
print('eval rows:', len(eval_dataset))
if len(train_dataset):
    sample_feature_schema = train_dataset[0].get('feature_schema_version', 'unknown') if hasattr(train_dataset[0], 'get') else 'unknown'
    print('feature schema:', sample_feature_schema)
print('sample prompt:\n', train_dataset[0]['text'][:500])

print('dataset preprocessing ready; run the next cell to start training')


In [ ]:
print('starting training cell; re-run previous cell if you changed dataset/preprocessing settings or want a fresh model state')
import inspect
if CFG['enable_wandb']:
    project_name = CFG['wandb_project'] or f"{task_type}_optimization"
    wandb.init(project=project_name, name=f"{task_type}_{CFG['model_type']}_r{CFG['lora_r']}", config=CFG)
else:
    os.environ['WANDB_MODE'] = 'disabled'

with open(output_dir / 'training_hyperparams_args.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    for k, v in CFG.items():
        writer.writerow([k, v])


if CFG['train_lm_head'] and CFG['train_embed_tokens']:
    from unsloth import UnslothTrainer, UnslothTrainingArguments
    Trainer = UnslothTrainer
    TrainingArgs = UnslothTrainingArguments
    print('Training with UnslothTrainer')
else:
    from trl import SFTTrainer
    from transformers import TrainingArguments
    Trainer = SFTTrainer
    TrainingArgs = TrainingArguments
    print('Training with SFTTrainer')

effective_bf16 = CFG['bf16'] if CFG['bf16'] is not None else (True if CFG['dtype'] == 'bfloat16' else False)
effective_fp16 = CFG['fp16'] if CFG['fp16'] is not None else (True if CFG['dtype'] == 'float16' else False)

training_args_kwargs = {
    'per_device_train_batch_size': CFG['per_device_train_batch_size'],
    'gradient_accumulation_steps': CFG['gradient_accumulation_steps'],
    'warmup_steps': CFG['warmup_steps'],
    'num_train_epochs': CFG['num_train_epochs'],
    'learning_rate': CFG['learning_rate'],
    'bf16': effective_bf16,
    'fp16': effective_fp16,
    'logging_steps': CFG['logging_steps'],
    'optim': CFG['optim'],
    'weight_decay': CFG['weight_decay'],
    'lr_scheduler_type': CFG['lr_scheduler_type'],
    'seed': CFG['seed'],
    'output_dir': str(output_dir),
    'report_to': (CFG['report_to'] if CFG['enable_wandb'] else 'none'),
    'load_best_model_at_end': (CFG['load_best_model_at_end'] if enable_eval else False),
    'metric_for_best_model': CFG['metric_for_best_model'],
    'greater_is_better': CFG['greater_is_better'],
    'save_total_limit': CFG['save_total_limit'],
    'save_steps': CFG['save_steps'],
    'save_strategy': CFG['save_strategy'],
    'eval_strategy': eval_strategy_value,
    'eval_steps': eval_steps_value,
    'per_device_eval_batch_size': CFG['per_device_eval_batch_size'],
    'group_by_length': CFG['group_by_length'],
    'dataloader_num_workers': CFG['dataloader_num_workers'],
    'remove_unused_columns': CFG['remove_unused_columns'],
    'run_name': CFG['run_name'],
}

supported_training_arg_names = {
    name for name in inspect.signature(TrainingArgs.__init__).parameters.keys()
    if name != 'self'
}

if 'eval_strategy' in training_args_kwargs and 'evaluation_strategy' in supported_training_arg_names:
    training_args_kwargs['evaluation_strategy'] = training_args_kwargs.pop('eval_strategy')
elif 'evaluation_strategy' in training_args_kwargs and 'eval_strategy' in supported_training_arg_names:
    training_args_kwargs['eval_strategy'] = training_args_kwargs.pop('evaluation_strategy')

final_training_args = {
    k: v for k, v in training_args_kwargs.items()
    if v is not None and k in supported_training_arg_names
}
dropped_training_args = sorted(
    k for k, v in training_args_kwargs.items()
    if v is not None and k not in supported_training_arg_names
)
if dropped_training_args:
    print('TrainingArgs unsupported keys dropped:', dropped_training_args)
print('resolved training args subset:', {
    'per_device_train_batch_size': final_training_args.get('per_device_train_batch_size'),
    'gradient_accumulation_steps': final_training_args.get('gradient_accumulation_steps'),
    'per_device_eval_batch_size': final_training_args.get('per_device_eval_batch_size'),
    'num_train_epochs': final_training_args.get('num_train_epochs'),
    'learning_rate': final_training_args.get('learning_rate'),
    'save_steps': final_training_args.get('save_steps'),
})

callbacks = []
if CFG.get('upload_on_save', False):
    callbacks.append(
        UploadCheckpointToHFCallback(
            repo_id=CFG['hf_model_repo_out'],
            token=HF_TOKEN,
            output_dir=output_dir,
            upload_source=CFG.get('upload_source', 'latest_checkpoint'),
        )
    )
    print(f"checkpoint auto-upload enabled -> {CFG['hf_model_repo_out']}")
else:
    print('checkpoint auto-upload disabled')

trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset_for_trainer,
    dataset_text_field='text',
    max_seq_length=CFG['max_seq_length'],
    dataset_num_proc=CFG['dataset_num_proc'],
    packing=False,
    args=TrainingArgs(**final_training_args),
    callbacks=callbacks if callbacks else None,
)
print('DEBUG: trainer object created')
resume_arg = None

if CFG['resume_from_checkpoint']:
    checkpoint_path = os.path.expanduser(CFG['resume_from_checkpoint'])
    if os.path.exists(checkpoint_path):
        print('resume from checkpoint:', checkpoint_path)
        resume_arg = checkpoint_path
    else:
        print('checkpoint not found, train from scratch:', checkpoint_path)
else:
    contains_checkpoints = any('checkpoint' in p.name and p.is_dir() for p in output_dir.iterdir())
    if contains_checkpoints:
        print('checkpoint detected in output_dir, continue training')
        resume_arg = True
    else:
        print('no checkpoint in output_dir, train from scratch')

print('DEBUG: building train dataloader', flush=True)
train_dataloader = trainer.get_train_dataloader()
print('DEBUG: train dataloader ready', flush=True)

print('DEBUG: fetching first batch', flush=True)
first_batch = next(iter(train_dataloader))
debug_batch_shapes = {}
for k, v in first_batch.items():
    if hasattr(v, 'shape'):
        debug_batch_shapes[k] = tuple(v.shape)
    else:
        debug_batch_shapes[k] = type(v).__name__
print('DEBUG: first batch fetched', debug_batch_shapes, flush=True)

del first_batch
del train_dataloader
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('DEBUG: entering trainer.train()', 'resume_arg=', resume_arg, flush=True)

if resume_arg is None:
    trainer.train()
else:
    trainer.train(resume_from_checkpoint=resume_arg)

final_dir = output_dir / 'final'
final_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(str(final_dir))

print('training done')
print('output_dir:', output_dir)
print('final_dir:', final_dir)
print('checkpoints:')
for p in sorted(output_dir.glob('checkpoint-*')):
    print(' -', p)


## 4) (선택) 학습 완료 모델 HF 업로드


In [ ]:
if CFG['upload_after_train']:
    api = HfApi(token=HF_TOKEN)
    api.create_repo(repo_id=CFG['hf_model_repo_out'], repo_type='model', private=False, exist_ok=True)

    output_dir = Path(resolve_output_dir(CFG, resolve_task_type(CFG)))
    upload_dir = resolve_upload_dir(output_dir, CFG['upload_source'], CFG['checkpoint_tag'])
    print('upload_dir:', upload_dir)

    upload_folder(
        repo_id=CFG['hf_model_repo_out'],
        repo_type='model',
        folder_path=str(upload_dir),
        token=HF_TOKEN,
        commit_message=f"Upload LoRA ({upload_dir.name}) from colab_02_full",
    )

    files = api.list_repo_files(repo_id=CFG['hf_model_repo_out'], repo_type='model')
    print(f"upload done: {CFG['hf_model_repo_out']} ({len(files)} files)")
    for f in files:
        print(' -', f)
else:
    print('CFG[upload_after_train]=False, skip upload')
